# Do bounded confidence or a co-evolving network change the controversy-axis result?

Two A/B backtests for [Lightningfish](https://github.com/rajul-kk/LightningFish) on Kaggle's free T4 GPU. Same setup as `kaggle_controversy.ipynb` (qwen2.5:7b via Ollama, no API key, $0) and the same population size (24 agents, 3-4 rounds), what fits a T4's VRAM without babysitting it.

METHODOLOGY.md's calibrated controversy run found crowd-split prediction below chance: 38% against a 53% baseline, n=74. Two mechanisms got proposed to make the split more realistic. Bounded confidence (Hegselmann-Krause) gates T3's herding update on `confidence_bound`, ignoring a target too far from an agent's own opinion. Already run (below): a non-effect, 48% gate off vs 52% on, both below the 62% baseline, neither significant (p=0.98, p=0.92). The 4-point gap is a coin flip's worth of per-event churn, not a real effect, logged as such in METHODOLOGY.md.

The second mechanism, a co-evolving follower network, hasn't been tested yet, this notebook adds it. Agents drop a followed peer once it drifts too far and refill from someone closer, so echo chambers form dynamically instead of being fixed at round 0 (`rewire_follower_graph`, opt-in via `coevolving_network=True`).

Neither is a claim the mechanism "works", these are mechanism tests. Every axis on this domain has failed the ladder so far, including bounded confidence, so the honest prior is the network arm probably won't move it either. Reporting whatever it does plainly is the point.


---
## What the data is

Same source as every HN backtest here: the [Algolia API](https://hn.algolia.com/api), free and unauthenticated (~10k req/hr), no key or scraping needed.

Sample: settled stories at least 24h old. `PULL_LIMIT` has to be generous, a local test at `limit=60` only cleared 20 of 60 scoreable (33%), under the harness's 15/15 minimum, so this pulls 250 for real margin.

Each seed is strictly submission-time fields: title, author/karma, url domain, type, self-text, never the outcome. Enforced in code and tested, not just convention.

Label: `num_comments / points` at settlement, 0.7+ is contested, under 0.4 is consensus, the gap (or under 20 points) is skipped. `kaggle_controversy.ipynb` has the full rationale for the cutoff.


---
## What's different between the arms

All three runs pull the same events, use the same calibration/evaluation split (a hash of the event id), and derive their threshold the same way. Only the flag passed to `_run_hn_controversy_calibrated` differs:

| Run | bounded_confidence | coevolving_network |
|---|---|---|
| `hn-controversy-calibrated` | True (default) | False (default) |
| `hn-controversy-calibrated-nobc` | False | False |
| `hn-controversy-calibrated-network` | True (default) | True |

The network arm's control is the first row, already run in the bounded-confidence test. The cache keys on both flags (`:bc1`, `:net1`), so this notebook only simulates the new network-enabled arm.


---
## 1. Setup

Sidebar: **Accelerator → GPU**, **Internet → On**.

Install `zstd` before Ollama: its installer needs it to extract, Kaggle's image doesn't ship it, and skipping this fails the install silently, surfacing later as a confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")


In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")


In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest yfinance

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms bounded confidence, the
# CachingAdapter kwarg-forwarding fix, and the calibrated-threshold code are
# actually present in this clone (all pushed in commit 17bcc07).
!python -m pytest tests/core tests/hn -q 2>&1 | tail -5


---
## 2. Configuration


In [ ]:
PULL_LIMIT = 250      # stories to pull; expect roughly 1 in 3 to be scoreable
N_AGENTS   = 24        # matches kaggle_controversy.ipynb's validated GPU size
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}, two arms")


### Throughput check

~26 model calls per event, times two arms back to back. Confirm the per-call cost before committing, double digits means you're on CPU regardless of the GPU assertion above.


In [ ]:
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event  ->  ~{per_call*26*2:.0f}s per event-pair (both arms)")


---
## 3. Run all three arms

The first two are the bounded-confidence A/B, summarized above. The third adds the co-evolving-network arm, using the first row as its control.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_on.log


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-nobc {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_off.log


### Co-evolving network arm

Same events, same split, `bounded_confidence` held at its default (True). The only variable: whether the follower graph rewires each round.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-network {PULL_LIMIT} 2>&1 | tee /kaggle/working/network_on.log


### Reading the logs

Each log ends with the standard report block (`beats_baselines`, `p_value_vs_best`) and along the way prints how many events survived the controversy-direction filter and the calibration/evaluation split. Those two numbers should match across all three logs (same event set); only the calibrated threshold is likely to differ slightly.

For bounded confidence (`bc_on.log` vs `bc_off.log`), the result's already in: a non-effect, discussed above. Re-running should land near 48%/52% again; at n=42 each event is worth ~2.4 points, so don't read a different verdict into it.

For the network arm (`network_on.log` against `bc_on.log`), two outcomes. Materially higher accuracy, or `beats_baselines` flipping to PASS, is a real signal worth carrying forward, pending a bigger n to rule out the same noise bounded confidence turned out to be. About the same or worse is consistent with the rest of this domain: HN reception is driven by who posts and replies early, not the crowd's social mechanic. Either way, log it in METHODOLOGY.md next to the bounded-confidence rows, and diff the two reports event by event before trusting a raw accuracy difference.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache


All three logs and the run cache land in `/kaggle/working/`, grab them from the Output tab. The cache keys each run by which flags were on (`:bc1`, `:net1`), so re-scoring later costs nothing.

Whatever the network arm turns up belongs in [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md) next to the bounded-confidence rows, a negative is exactly as worth recording as the others.
